In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
import os

torch.mps.empty_cache()

# ======================
# CONFIG
# ======================
DATA_DIR = "dataset"
BATCH_SIZE = 32
EPOCHS_HEAD = 8
EPOCHS_FINE = 25
NUM_CLASSES = 60  # change this

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# ======================
# TRANSFORMS
# ======================
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(300, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),

    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.2,
        saturation=0.2,
        hue=0.03
    ),

    transforms.ToTensor(),

    transforms.RandomErasing(p=0.2),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ======================
# DATA
# ======================
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# ======================
# MODEL
# ======================

weights = EfficientNet_B3_Weights.DEFAULT
model = efficientnet_b3(weights=weights)

# Replace classifier
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(DEVICE)

# ======================
# LOSS
# ======================
criterion = nn.CrossEntropyLoss()

# ======================
# TRAIN FUNCTION
# ======================
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct = 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

# ======================
# VALIDATION FUNCTION
# ======================
def validate(model, loader):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

# ======================
# EARLY STOPPING
# ======================
best_val_acc = 0
patience = 5
counter = 0

def check_early_stop(val_acc):
    global best_val_acc, counter
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        counter += 1
    return counter >= patience

# ======================
# PHASE 1: TRAIN HEAD
# ======================
for param in model.features.parameters():
    param.requires_grad = False

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print("=== Training Head ===")
for epoch in range(EPOCHS_HEAD):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_acc = validate(model, val_loader)

    print(f"[Head] Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

# ======================
# PHASE 2: FINE-TUNE
# ======================
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINE)

print("=== Fine-Tuning ===")
for epoch in range(EPOCHS_FINE):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_acc = validate(model, val_loader)

    scheduler.step()

    print(f"[Fine] Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

    if check_early_stop(val_acc):
        print("Early stopping triggered.")
        break

print("Training complete. Best model saved as best_model.pth")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
# from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import gc

import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


torch.mps.empty_cache()

# ======================
# CONFIG
# ======================
DATA_DIR = "dataset"
BATCH_SIZE = 4
EPOCHS = 8   # 👈 small epoch for testing
LR = 1e-4
IMAGE_SIZE = 224 # 224 b0, 300 b3

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(DEVICE)

# ======================
# TRANSFORMS
# ======================
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(0.3, 0.2, 0.2, 0.03),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ======================
# DATASET
# ======================
train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_tf)
val_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=val_tf)

# ======================
# BALANCED SAMPLER
# ======================
targets = np.array(train_ds.targets)
class_counts = np.bincount(targets)

class_weights = 1.0 / (class_counts + 1e-6)
sample_weights = class_weights[targets]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    num_workers=0,
    # shuffle=True
    sampler=sampler
)#num_workers=4)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# ======================
# MODEL
# ======================

# weights = EfficientNet_B3_Weights.DEFAULT
# model = efficientnet_b3(weights=weights)

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    len(train_ds.classes)
)

model = model.to(DEVICE)

# ======================
# LOSS & OPTIMIZER
# ======================
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# ======================
# CONFUSION MATRIX
# ======================
def evaluate_with_confusion_matrix(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            outputs = model(x)
            preds = outputs.argmax(1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y.numpy())

            

    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig("confusion_matrix.png")
    plt.close()

    print("Saved confusion_matrix.png")

    # Per-class accuracy
    acc = cm.diagonal() / cm.sum(axis=1)
    for i, cls in enumerate(train_ds.classes):
        print(f"{cls}: {acc[i]:.2f}")

# ======================
# TRAIN LOOP
# ======================
from collections import defaultdict
def train():
    for epoch in range(EPOCHS):

        model.train()
        correct = 0
        total_loss = 0

        class_sample_count = defaultdict(int)

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            # ✅ Count labels in this batch
            for label in y.cpu().numpy():
                class_sample_count[label] += 1

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            correct += (out.argmax(1) == y).sum().item()
            
            print(f"Epoch {epoch+1}/{EPOCHS} | "
                  f"Batch Loss: {loss.item():.4f} | "
                  f"Batch Acc: {(out.argmax(1) == y).float().mean().item():.4f}", end="\r")

            
        train_acc = correct / len(train_ds)


        print("\nSampled per class this epoch:")
        for idx, count in class_sample_count.items():
            class_name = train_ds.classes[idx]
            print(f"{class_name}: {count}")

        # validation
        model.eval()
        val_correct = 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                val_correct += (out.argmax(1) == y).sum().item()

        val_acc = val_correct / len(val_ds)

        print(f"Epoch {epoch+1}/{EPOCHS} | "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
        

    torch.save(model.state_dict(), "efficientnet_b3_test.pth")
    print("Saved model: efficientnet_b3_test.pth")
    
    # ======================
    # CONFUSION MATRIX
    # ======================
    evaluate_with_confusion_matrix(model, val_loader)

# ======================
# RUN
# ======================
if __name__ == "__main__":
    train()

mps
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2]
[40 40 40]
[0.025 0.025 0.025]
[0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025
 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.025 0.

KeyboardInterrupt: 

In [12]:
import torch
torch.mps.empty_cache()
def print_mps_memory():
    allocated = torch.mps.current_allocated_memory() / 1024**2
    reserved = torch.mps.driver_allocated_memory() / 1024**2
    print(f"MPS Allocated: {allocated:.2f} MB | Reserved: {reserved:.2f} MB")

print_mps_memory()

MPS Allocated: 64.21 MB | Reserved: 1192.80 MB


In [ ]:
import torch
import torch.nn as nn
# from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# rebuild model
# weights = EfficientNet_B3_Weights.DEFAULT
# model = efficientnet_b3(weights=None)  # ⚠️ no pretrained weights here

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)


model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES  # 👈 must match training
)

model.load_state_dict(torch.load("efficientnet_b3_test.pth", map_location=DEVICE))
model = model.to(DEVICE)
model.eval()